In [ ]:
import eradiate
from eradiate.units import unit_registry as ureg
import mitsuba as mi

eradiate.set_mode("mono")

In [ ]:
import numpy as np
import xarray as xr

In [ ]:
# Downsample GEBCO data for optimal performance
with xr.open_dataset(
    "data/GEBCO_16_May_2025_6841353ff35f/gebco_2024_sub_ice_n73.0_s33.0_w-12.0_e32.0.nc"
).astype("float64") as ds:
    ds.interp(
        lon=np.linspace(ds["lon"].min(), ds["lon"].max(), 441),
        lat=np.linspace(ds["lat"].min(), ds["lat"].max(), 401),
    ).dropna("lon", how="all").dropna("lat", how="all").to_netcdf(
        "data/gebco_europe_downsampled.nc"
    )

In [ ]:
# This is our comparison reference
from eradiate.scenes.surface import mesh_from_dem

## Convert lonlat indexed data to a PLY mesh
da = xr.load_dataset("data/gebco_europe_downsampled.nc")["elevation"]
da["lon"].attrs["units"] = "deg"
da["lat"].attrs["units"] = "deg"
mesh, lon_lim, lat_lim = mesh_from_dem(
    da, geometry="spherical_shell", add_texcoords=True
)
mesh.write_ply("data/gebco_europe_downsampled_rotated.ply")
del mesh

## Load this into a DEMExperiment
exp = eradiate.experiments.DEMExperiment(
    geometry="spherical_shell",
    surface={
        "type": "dem",
        "construct": "from_mesh",
        "geometry": "spherical_shell",
        "xlon_lim": lon_lim,
        "ylat_lim": lat_lim,
        "mesh": eradiate.scenes.shapes.shape_factory.convert(
            {
                "type": "file_mesh",
                "filename": "data/gebco_europe_downsampled_rotated.ply",
            }
        ),
        "bsdf_mesh": {
            "type": "lambertian",
            "reflectance": 0.75,
        },
    },
    atmosphere=None,
    illumination={"type": "constant"},
    measures={
        "type": "perspective",
        "film_resolution": (512, 512),
        "origin": [0, 0, 15000] * ureg.km,
        "target": [0, 0, 0],
        "up": [0, 1, 0],
    },
)
result = eradiate.run(exp, spp=16)
result.squeeze()["radiance"].plot.imshow()


In [ ]:
from eradiate.scenes.geometry import SceneGeometry
from eradiate.units import to_quantity
from eradiate.scenes.surface import triangulate_grid
from eradiate.scenes.shapes import BufferMeshShape


def _middle(a, axis=None):
    a_min = a.min(axis=axis)
    a_max = a.max(axis=axis)
    return 0.5 * (a_min + a_max)


def _dem_texcoords(xlon, ylat, xlon_lim, ylat_lim) -> np.ndarray:
    uvs_x = (xlon - xlon_lim[0]) / (xlon_lim[1] - xlon_lim[0])
    uvs_y = (ylat - ylat_lim[0]) / (ylat_lim[1] - ylat_lim[0])
    return np.stack([uvs_x, uvs_y]).T


def _transform_vertices_spherical_shell_lonlat(vertices, planet_radius):
    """
    Convert the (lon, lat, elevation) vertices from the initial vertex generation
    into (x, y, z) values for spherical shell geometries.

    Parameters
    ----------
    vertices : ndarray
        List of mesh vertices in (lon, lat, elevation) tuples, with lon and lat
        in radians.

    planet_radius : float
        Planet radius in kernel length units.

    Returns
    -------
    vertices : ndarray
    """
    lon = vertices[:, 0]
    lat = vertices[:, 1]
    lon_center, lat_center = _middle(vertices[:, :2], axis=0)
    elevation = vertices[:, 2]

    phi_r = lon
    theta_r = np.pi / 2.0 - lat

    x = np.sin(theta_r) * np.cos(phi_r) * (elevation + planet_radius)
    y = np.sin(theta_r) * np.sin(phi_r) * (elevation + planet_radius)
    z = np.cos(theta_r) * (elevation + planet_radius)
    vertices = np.array((x, y, z)).transpose()

    # # At this point, vertices are positioned in an ECEF frame; transform them to
    # # the local ENU frame
    # trafo = _transform_lonlat_range_to_local(lon_center, lat_center)
    # vertices = _apply(trafo, vertices)
    return vertices


def mesh_from_dem(da, geometry, add_texcoords=True):
    # Pre-process geometry parameter
    geometry = SceneGeometry.convert(geometry)

    # Set default planet radius value
    if isinstance(geometry, eradiate.scenes.geometry.SphericalShellGeometry):
        planet_radius = geometry.planet_radius
    else:
        raise TypeError(f"{geometry = }")

    # Check data array coordinates and dimensions
    length_kernel_u = ureg("m")

    mode = "lonlat"
    xlon_dim, ylat_dim = "lon", "lat"

    # Convert to distances in plane-parallel geometry,
    # and to angles in spherical-shell geometry.
    xlon = to_quantity(da[xlon_dim])
    ylat = to_quantity(da[ylat_dim])
    # Extract elevation data and ensure y-major layout
    elevation = to_quantity(da.transpose(xlon_dim, ylat_dim))
    # By default, no texture coordinates are assigned
    texcoords = None

    # Process vertex data depending on geometry and coordinate mode, generate triangulation
    if mode == "lonlat":
        xlon = xlon.to(ureg.rad)
        ylat = ylat.to(ureg.rad)
        vertices, faces = triangulate_grid(
            xlon.m, ylat.m, elevation.m_as(length_kernel_u)
        )

        # If relevant, assign texture coordinates
        xlon_lim = (xlon.m.min(), xlon.m.max()) * xlon.u
        ylat_lim = (ylat.m.min(), ylat.m.max()) * ylat.u
        if add_texcoords:
            texcoords = _dem_texcoords(
                vertices[:, 0], vertices[:, 1], xlon_lim.m, ylat_lim.m
            )

        # Rotate mesh to Eradiate's local frame (located at the North pole)
        vertices = _transform_vertices_spherical_shell_lonlat(
            vertices, planet_radius.m_as(length_kernel_u)
        )
        # Recompute limits (returned afterwards)
        xlon_lim = (xlon.m.min(), xlon.m.max()) * xlon.u
        ylat_lim = (ylat.m.min(), ylat.m.max()) * ylat.u

    else:
        raise RuntimeError(f"unknown input mode {mode}")

    # Create mesh instance
    mesh = BufferMeshShape(vertices=vertices, faces=faces, texcoords=texcoords)

    return mesh, xlon_lim, ylat_lim


da = xr.load_dataset("data/gebco_europe_downsampled.nc")["elevation"]
da["lon"].attrs["units"] = "deg"
da["lat"].attrs["units"] = "deg"
mesh, lon_lim, lat_lim = mesh_from_dem(
    da, geometry="spherical_shell", add_texcoords=True
)
mesh.write_ply("data/gebco_europe_downsampled.ply")
del mesh


mi_scene = mi.load_dict(
    {
        "type": "scene",
        "dem": {
            "type": "ply",
            "filename": "data/gebco_europe_downsampled.ply",
        },
        "illumination": {"type": "constant"},
        "camera": {
            "type": "perspective",
            "far_clip": 1e12,
            "to_world": mi.ScalarTransform4f.look_at(
                origin=mi.Point3f(3465866.375, 861189.625, 4785356.0) * 2,
                target=[3465866.375, 861189.625, 4785356.0],
                up=[0, 0, 1],
            ),
        },
        "integrator": {"type": "volpath"},
    }
)
mi.Bitmap(mi.render(mi_scene))

In [ ]:
def to_uv(lon_lim, lat_lim):
    """
    Compute the `to_uv` transformation for the opacity mask bitmap.
    It moves the central (transparent) part of the bitmap to where
    it covers the specified longitude/latitude extent.
    """
    lon_range = lon_lim[1] - lon_lim[0]
    lon_scale = 120.0 / lon_range

    lat_range = lat_lim[1] - lat_lim[0]
    lat_scale = 60.0 / lat_range

    lon_middle = _middle(lon_lim)
    lon_uv = lon_middle / 360.0 + 0.5

    lat_mean = (lat_lim[1] + lat_lim[0]) / 2.0
    lat_uv = 0.5 - (lat_mean / 180)

    return mi.ScalarTransform4f.scale(
        (lon_scale, lat_scale, 1.0)
    ) @ mi.ScalarTransform4f.translate(
        [-lon_uv + (0.5 / lon_scale), -lat_uv + (0.5 / lat_scale), 0.0]
    )


# opacity_to_uv = to_uv(lon_lim.m_as(ureg.deg), lat_lim.m_as(ureg.deg))
opacity_to_uv = to_uv(np.array([-30, 30]), np.array([-60, 60]))
opacity_bitmap = mi.Bitmap(
    np.array(
        [
            [1.0, 1.0, 1.0],
            [1.0, 0.0, 1.0],
            [1.0, 1.0, 1.0],
        ],
    )
)

mask_dict = {
    "type": "mask",
    "opacity": {
        "type": "bitmap",
        "bitmap": opacity_bitmap,
        "filter_type": "nearest",
        "wrap_mode": "clamp",
        "to_uv": opacity_to_uv,
    },
    "nested_bsdf": {
        "type": "diffuse",
        "reflectance": {"type": "uniform", "value": 0.5},
    },
}

mi_scene = mi.load_dict(
    {
        "type": "scene",
        "sphere": {
            "type": "sphere",
            "bsdf": mask_dict,
        },
        "illumination": {"type": "constant"},
        "camera": {
            "type": "perspective",
            "to_world": mi.ScalarTransform4f.look_at(
                origin=[0, 0, 5], target=[0, 0, 0], up=[0, 1, 0]
            ),
        },
        "integrator": {"type": "volpath"},
    }
)

mi.Bitmap(mi.render(mi_scene))

In [ ]:
from eradiate.constants import EARTH_RADIUS


opacity_to_uv = to_uv(lon_lim.m_as(ureg.deg), lat_lim.m_as(ureg.deg))
opacity_bitmap = mi.Bitmap(
    np.array(
        [
            [1.0, 1.0, 1.0],
            [1.0, 0.0, 1.0],
            [1.0, 1.0, 1.0],
        ],
    )
)
mask_dict = {
    "type": "mask",
    "opacity": {
        "type": "bitmap",
        "bitmap": opacity_bitmap,
        "filter_type": "nearest",
        "wrap_mode": "clamp",
        "to_uv": opacity_to_uv,
    },
    "nested_bsdf": {
        "type": "diffuse",
        "reflectance": {"type": "uniform", "value": 0.5},
    },
}

mi_scene = mi.load_dict(
    {
        "type": "scene",
        "dem": {
            "type": "ply",
            "filename": "data/gebco_europe_downsampled.ply",
        },
        "background_sphere": {
            "type": "sphere",
            "bsdf": mask_dict,
            "to_world": mi.ScalarTransform4f.rotate([0, 0, 1], 180)  # Might indicate a bug in the uv coords computation
            @ mi.ScalarTransform4f.scale(EARTH_RADIUS.m_as("m") * 0.99),
        },
        "illumination": {"type": "constant"},
        "camera": {
            "type": "perspective",
            "far_clip": 1e12,
            "to_world": mi.ScalarTransform4f.look_at(
                origin=mi.Point3f(0, 0, 30e6),
                target=[0, 0, 0],
                up=[0, 1, 0],
            ),
        },
        "integrator": {"type": "volpath"},
    }
)

mi.Bitmap(mi.render(mi_scene))

In [ ]:
# Create mesh and visualize it
kdict = {
    "europe_dem": {
        "type": "ply",
        "filename": "data/gebco_europe_downsampled.ply",
        "bsdf": {"type": "diffuse", "reflectance": 0.75},
    }
}  # TBD: Add background sphere
kpmap = {}  # TBD: Add update protocols for DEM and background sphere

exp = eradiate.experiments.AtmosphereExperiment(
    geometry="spherical_shell",
    surface=None,
    atmosphere=None,
    illumination={"type": "constant"},
    measures={
        "type": "perspective",
        "film_resolution": (512, 512),
        "far_clip": 1e20,
        "origin": [0, 0, 30000] * ureg.km,
        "target": [0, 0, 0],
        "up": [0, 1, 0],
    },
    kdict=kdict,
)
result = eradiate.run(exp, spp=16)
result.squeeze()["radiance"].plot.imshow()
